# 📄 Aula 2 — Docling: Ingestão Inteligente de PDFs Jurídicos
## MBA em RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Dominar a ingestão de PDFs complexos com Docling, comparar com PyPDF2 e integrar ao pipeline RAG.

**Duração estimada:** 75 minutos

---
### Diagrama da Aula
```
PDF Jurídico Complexo
        │
        ├──▶ PyPDF2 (baseline)     ──▶ texto plano sem estrutura
        │
        └──▶ Docling (avançado)    ──▶ Markdown estruturado
                  │                        + tabelas extraídas
                  │                        + metadados de seção
                  │                        + suporte a OCR
                  │
                  └──▶ LangChain Documents com metadados ricos
```

> **ABNT:** IBM RESEARCH. *Docling: An Efficient Document Conversion Library*. GitHub, 2024.

## 🔧 Célula 1 — Instalação

O Docling tem dependências pesadas (modelos de layout). A instalação pode levar **3–5 minutos** no Colab.

> ⚠️ **Atenção:** O Docling baixa modelos (~500MB) na primeira execução. Isso é normal.

In [4]:
# ============================================================
# INSTALAÇÃO DO DOCLING E DEPENDÊNCIAS
# ============================================================

# Docling: biblioteca principal de ingestão
!pip install docling

# PyPDF2: para comparação (baseline)
!pip install PyPDF2

# Dependências adicionais
!pip install langchain langchain-community
!pip install requests  
# Fallback de PDFs
!pip install -q reportlab 

print("✅ Instalação concluída!")
print()
print("📦 Versões instaladas:")
import importlib.metadata
import docling
import PyPDF2

print(f"   docling: {importlib.metadata.version('docling')}")
print(f"   PyPDF2:  {PyPDF2.__version__}")

✅ Instalação concluída!

📦 Versões instaladas:
   docling: 2.94.0
   PyPDF2:  3.0.1


## 📥 Célula 2 — Datasets PDF da Aula 2 (uso de PDFs reais)

Em vez de gerar PDFs sintéticos com `reportlab`, este exemplo usa os **dois PDFs reais** que ficam em `aula2/datasets/`:

| Arquivo | Tipo | Comportamento esperado |
|---|---|---|
| `Manual_DPCA_atualizado.pdf` | PDF **digital** com texto extraível | Docling extrai estrutura sem OCR (rápido) |
| `Laudo.pdf` | PDF com **imagem de texto** (escaneado) | Docling precisa de `do_ocr=True` (lento, mas funciona) |

Esses são os datasets oficiais da Aula 2 — também usados pelo `LAB1_Docling_Ingestao_Avancada.ipynb`. Se você rodar este notebook fora do projeto (sem a pasta `datasets/`), o código abaixo gera um fallback sintético para que o exemplo continue executável.


In [15]:
# ============================================================
# PREPARAÇÃO: Usar os 2 PDFs reais do dataset da Aula 2
#   - Manual_DPCA_atualizado.pdf  → PDF digital (texto extraível)
#   - Laudo.pdf                    → PDF com imagem de texto (precisa OCR)
# Localização: aula2/datasets/ (relativo ao notebook em aula2/exemplos/)
# ============================================================
from pathlib import Path
import os

# Caminho do dataset
DATASET_DIR = Path("../datasets").resolve()
if not DATASET_DIR.exists():
    DATASET_DIR = Path.cwd().parent / "datasets"

PDF_MANUAL = DATASET_DIR / "Manual_DPCA_atualizado.pdf"
PDF_LAUDO  = DATASET_DIR / "Laudo.pdf"

print("="*60)
print("Datasets PDF da Aula 2")
print("="*60)
print(f"Dataset dir: {DATASET_DIR}")
print(f"  • Manual_DPCA_atualizado.pdf existe? {PDF_MANUAL.exists()}")
print(f"  • Laudo.pdf                  existe? {PDF_LAUDO.exists()}")

# Variáveis usadas pelas células seguintes (mantemos os nomes históricos)
# acordao_simples.pdf       → Manual_DPCA_atualizado.pdf  (digital)
# relatorio_inteligencia.pdf → Laudo.pdf                   (com OCR)
caminho_pdf_simples  = str(PDF_MANUAL) if PDF_MANUAL.exists() else None
caminho_pdf_complexo = str(PDF_LAUDO)  if PDF_LAUDO.exists()  else None

if caminho_pdf_simples is None or caminho_pdf_complexo is None:
    print("\n⚠️  Algum PDF do dataset não foi encontrado. Gerando fallback sintético...")
    
    # Fallback mínimo para quem rodar o notebook sem os PDFs reais
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import SimpleDocTemplate, Paragraph

    styles = getSampleStyleSheet()
    if caminho_pdf_simples is None:
        caminho_pdf_simples = "/tmp/fallback_simples.pdf"
        SimpleDocTemplate(caminho_pdf_simples, pagesize=A4).build([
            Paragraph("ACÓRDÃO Nº 1234/2025 - HABEAS CORPUS", styles["Heading1"]),
            Paragraph("Fallback gerado porque Manual_DPCA não foi encontrado.", styles["Normal"]),
        ])
    if caminho_pdf_complexo is None:
        caminho_pdf_complexo = "/tmp/fallback_complexo.pdf"
        SimpleDocTemplate(caminho_pdf_complexo, pagesize=A4).build([
            Paragraph("LAUDO PERICIAL", styles["Heading1"]),
            Paragraph("Fallback gerado porque Laudo.pdf não foi encontrado.", styles["Normal"]),
        ])

print(f"\n✓ caminho_pdf_simples  = {caminho_pdf_simples}")
print(f"✓ caminho_pdf_complexo = {caminho_pdf_complexo}")
print("\nNOTA: o caminho_pdf_complexo (Laudo.pdf) é um PDF escaneado → precisa OCR.")
print("       A célula que processa esse PDF habilita do_ocr=True.")


Datasets PDF da Aula 2
Dataset dir: D:\IBMEC\MBA - RAG & CAG Aplicados a Direito e Segurança Pública\aula2\datasets
  • Manual_DPCA_atualizado.pdf existe? True
  • Laudo.pdf                  existe? True

✓ caminho_pdf_simples  = D:\IBMEC\MBA - RAG & CAG Aplicados a Direito e Segurança Pública\aula2\datasets\Manual_DPCA_atualizado.pdf
✓ caminho_pdf_complexo = D:\IBMEC\MBA - RAG & CAG Aplicados a Direito e Segurança Pública\aula2\datasets\Laudo.pdf

NOTA: o caminho_pdf_complexo (Laudo.pdf) é um PDF escaneado → precisa OCR.
       A célula que processa esse PDF habilita do_ocr=True.


---
## 📉 Célula 3 — Extração com PyPDF2 (Baseline)

Antes de ver o Docling, vamos estabelecer o **baseline** com PyPDF2 para ter uma comparação justa.

In [6]:
# ============================================================
# PYPDF2: Extração básica (baseline para comparação)
# ============================================================
import PyPDF2

def extrair_pypdf2(caminho_pdf):
    """Extrai texto de um PDF usando PyPDF2."""
    with open(caminho_pdf, 'rb') as arquivo:
        reader = PyPDF2.PdfReader(arquivo)
        textos = []
        for i, pagina in enumerate(reader.pages):
            texto = pagina.extract_text()
            textos.append({
                "pagina": i + 1,
                "texto": texto,
                "chars": len(texto) if texto else 0
            })
    return textos

# Testando em ambos os PDFs
print("📉 PYPDF2 — EXTRAÇÃO BASELINE")
print("="*60)

print("\n[1] Acórdão Simples:")
resultado_simples = extrair_pypdf2(caminho_pdf_simples)
for pag in resultado_simples:
    print(f"  Página {pag['pagina']}: {pag['chars']} caracteres")
    print(f"  Amostra: {pag['texto'][:300].strip()!r}")

print("\n[2] Relatório de Inteligência (com tabela):")
resultado_complexo = extrair_pypdf2(caminho_pdf_complexo)
for pag in resultado_complexo:
    print(f"  Página {pag['pagina']}: {pag['chars']} caracteres")
    print(f"  Amostra: {pag['texto'][:500].strip()!r}")

print()
print("⚠️  OBSERVE: A tabela de estatísticas foi extraída como texto")
print("   confuso, sem preservar a estrutura linha × coluna.")

📉 PYPDF2 — EXTRAÇÃO BASELINE

[1] Acórdão Simples:
  Página 1: 150 caracteres
  Amostra: 'MANUAL  \nPARA USO DO PROTOCOLO DE POLÍCIA JUDICIÁRIA PARA DEPOIMENTO \nESPECIAL DE CRIANÇA E ADOLESCENTE  \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nABRIL /2019'
  Página 2: 1122 caracteres
  Amostra: '2 \n \n \nDiretor Geral da Polícia Civil do Distrito Federal:  \nDelegado de Polícia Robson Cândido da Silva  \n \nDiretor  do Departamento de Polícia  Especializada:  \nDelegado  de Polícia  Victor Dan de Alencar Alves  \n \nDelegada Chefe da Delegacia Especial  de Proteção  à Criança e ao Adolescente'
  Página 3: 3071 caracteres
  Amostra: '3 \nSumário  \n \n1. Apresentação……………………………………………………… .…………….. ......4 \n2. Introdução Geral………………………………………………… .………………. ......5 \n3. A Polícia Judiciária e a Delegacia de Proteção à Criança e ao Adolescente  (DPCA) \n…………………………………………………………….................................... .......7 \n4. Postura'
  Página 4: 2383 caracteres
  Amostra: '4 \n1. APRESENTAÇÃ 

---
## 🚀 Célula 4 — Extração com Docling (Tutorial Passo a Passo)

### Arquitetura do Docling

```
PDF Input
    │
    ▼
┌─────────────────────────────────────────────┐
│            DocumentConverter                │
│  ┌─────────────┐    ┌────────────────────┐  │
│  │ PDF Backend │───▶│  Pipeline Steps:   │  │
│  │(pypdfium2)  │    │  1. Layout Analysis│  │
│  └─────────────┘    │  2. Table Structure│  │
│                     │  3. OCR (opcional) │  │
│                     │  4. Reading Order  │  │
│                     └────────────────────┘  │
└─────────────────────────────────────────────┘
    │
    ▼
DoclingDocument
    │
    ├── .export_to_markdown()   → Markdown estruturado
    ├── .export_to_dict()       → JSON estruturado
    └── .tables                 → Tabelas extraídas
```

In [8]:
# ============================================================
# PASSO 1: Importar e configurar o DocumentConverter (Docling v2.x)
# ============================================================
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

# 1. Configuração específica para PDFs
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False              # False: mais rápido para PDFs com texto nativo
pipeline_options.do_table_structure = True   # ESSENCIAL para extrair tabelas

print("⏳ Inicializando Docling...")
print("   (Pode demorar 1–2 min para baixar modelos na primeira execução)")

# 2. Inicializa o DocumentConverter mapeando o formato PDF para suas opções correspondentes
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

print("✅ Docling inicializado com sucesso!")


⏳ Inicializando Docling...
   (Pode demorar 1–2 min para baixar modelos na primeira execução)
✅ Docling inicializado com sucesso!


In [17]:
# ============================================================
# PASSO 2: Converter o PDF digital (Manual_DPCA_atualizado.pdf)
# ============================================================
import time
from pathlib import Path
# 1. Definindo o caminho absoluto correto com raw string (r"...")
caminho_pdf_simples = Path(r"Manual_DPCA_atualizado.pdf")
print(f"Caminho configurado: {caminho_pdf_simples}")
print(f"O Python encontrou o arquivo? {caminho_pdf_simples.exists()}")

print(f"🔄 Convertendo: {caminho_pdf_simples}")
inicio = time.time()

# convert() retorna um ConversionResult
result_acordao = converter.convert(caminho_pdf_simples)

tempo = time.time() - inicio
print(f"✅ Conversão concluída em {tempo:.2f}s")

# Exportando para Markdown estruturado
markdown_acordao = result_acordao.document.export_to_markdown()

print()
print("📝 OUTPUT DOCLING — Manual_DPCA em Markdown (primeiros 1500 chars):")
print("="*60)
print(markdown_acordao[:1500])
print("[...]")

print(f"\n📊 Estatísticas:")
print(f"   • Tamanho do Markdown: {len(markdown_acordao)} chars")
print(f"   • Elementos detectados: {len(result_acordao.document.children)}")
print(f"   • Tabelas: {len(result_acordao.document.tables)}")


An unexpected error occurred while opening the document Manual_DPCA_atualizado.pdf
Traceback (most recent call last):
  File "d:\IBMEC\MBA - RAG & CAG Aplicados a Direito e Segurança Pública\.venv\Lib\site-packages\docling\datamodel\document.py", line 162, in __init__
    self._init_doc(backend, path_or_stream)
  File "d:\IBMEC\MBA - RAG & CAG Aplicados a Direito e Segurança Pública\.venv\Lib\site-packages\docling\datamodel\document.py", line 220, in _init_doc
    self._backend = backend(self, path_or_stream=path_or_stream)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\IBMEC\MBA - RAG & CAG Aplicados a Direito e Segurança Pública\.venv\Lib\site-packages\docling\backend\docling_parse_backend.py", line 232, in __init__
    self.parser = DoclingPdfParser(loglevel="fatal")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\IBMEC\MBA - RAG & CAG Aplicados a Direito e Segurança Pública\.venv\Lib\site-packages\docling_parse\pdf_parser.py", line 776, 

Caminho configurado: Manual_DPCA_atualizado.pdf
O Python encontrou o arquivo? True
🔄 Convertendo: Manual_DPCA_atualizado.pdf


ConversionError: Input document Manual_DPCA_atualizado.pdf is not valid.

In [ ]:
# ============================================================
# PASSO 3: Converter o PDF escaneado (Laudo.pdf — IMAGEM DE TEXTO)
# Este caso REQUER OCR (do_ocr=True). É onde o Docling brilha contra o PyPDF2.
# ============================================================
print(f"🔄 Convertendo: {caminho_pdf_complexo}")
print("   Este PDF é escaneado (imagem de texto) — habilitando do_ocr=True.")
print("   ⏳ Pode levar 30s–3min dependendo do número de páginas e CPU.")

# Reconfigurar o converter com OCR ATIVO
pipeline_options_ocr = PipelineOptions(
    do_ocr=True,                # ATIVA o motor OCR (EasyOCR)
    do_table_structure=True,    # mantém reconstrução de tabelas, se houver
)
converter_ocr = DocumentConverter(pipeline_options=pipeline_options_ocr)

inicio = time.time()
try:
    result_relatorio = converter_ocr.convert(caminho_pdf_complexo)
    tempo = time.time() - inicio
    print(f"✅ Conversão (com OCR) concluída em {tempo:.2f}s")

    markdown_relatorio = result_relatorio.document.export_to_markdown()

    print()
    print("📝 OUTPUT DOCLING — Laudo.pdf em Markdown (primeiros 2000 chars):")
    print("="*60)
    print(markdown_relatorio[:2000])
    print("[...]")
    print()
    print(f"📊 Estatísticas:")
    print(f"   • Tamanho do Markdown:  {len(markdown_relatorio)} chars")
    print(f"   • Elementos detectados: {len(result_relatorio.document.children)}")
    print(f"   • Tabelas detectadas:    {len(result_relatorio.document.tables)}")
    print()
    print("🎯 OBSERVE: o Docling RECONSTRUIU texto a partir de uma imagem!")
    print("   PyPDF2 (sem OCR) teria retornado string vazia.")
except Exception as e:
    print(f"\n❌ Erro no processamento OCR: {e}")
    print("   • Verifique se EasyOCR foi baixado (1ª execução demora)")
    print("   • Em máquinas sem GPU, OCR é dezenas de vezes mais lento")
    result_relatorio = None
    markdown_relatorio = ""


In [ ]:
# ============================================================
# PASSO 4: Acessando elementos estruturados do documento
# O DoclingDocument é um objeto rico — muito mais que texto plano
# ============================================================
print("🏗️ ESTRUTURA DO DOCLING DOCUMENT")
print("="*60)

doc = result_relatorio.document

# Iterando pelos elementos do documento com seus tipos
print("\n📋 Elementos identificados:")
contagem_tipos = {}
for elemento in doc.iterate_items():
    # elemento é uma tupla (item, level) em versões recentes do Docling
    item = elemento[0] if isinstance(elemento, tuple) else elemento
    tipo = type(item).__name__
    contagem_tipos[tipo] = contagem_tipos.get(tipo, 0) + 1

for tipo, count in sorted(contagem_tipos.items()):
    print(f"   {tipo}: {count}")

print()
# Acessando tabelas especificamente
print(f"📊 Tabelas identificadas: {len(doc.tables)}")
for i, tabela in enumerate(doc.tables):
    print(f"\nTabela {i+1}:")
    print(f"  Linhas: {tabela.num_rows}")
    print(f"  Colunas: {tabela.num_cols}")
    # Exporta tabela como DataFrame pandas
    try:
        df_tabela = tabela.export_to_dataframe()
        print(f"  Preview:\n{df_tabela.to_string()}")
    except Exception as e:
        print(f"  (Erro ao exportar DataFrame: {e})")

---
## 🔄 Célula 5 — Comparação Docling vs PyPDF2

Vamos quantificar as diferenças de forma objetiva:

In [ ]:
# ============================================================
# COMPARAÇÃO QUANTITATIVA: PyPDF2 vs Docling
# ============================================================
import pandas as pd

# Extração PyPDF2
pypdf_resultado = extrair_pypdf2(caminho_pdf_complexo)
texto_pypdf = " ".join([p["texto"] for p in pypdf_resultado if p["texto"]])

# Extração Docling
texto_docling = markdown_relatorio

# Verificações qualitativas
def verifica_qualidade(texto, nome):
    verificacoes = {
        "Ferramenta": nome,
        "Total chars": len(texto),
        "Total palavras": len(texto.split()),
        "Tem 'RELATÓRIO'": 'RELATÓRIO' in texto.upper() or 'RELATORIO' in texto.upper(),
        "Tem 'ESTATÍSTICAS'": 'ESTATÍSTICAS' in texto.upper() or 'ESTATISTICAS' in texto.upper(),
        "Tem tabela (|)": '|' in texto,  # Markdown table indicator
        "Tem Headers (#)": '#' in texto,
        "Tem '113' (total tráfico)": '113' in texto,
        "Tem '11,8%' (variação)": '11,8%' in texto or '11.8%' in texto,
        "Tem seções identificadas": texto.count('\n## ') > 0 or texto.count('\n# ') > 0,
    }
    return verificacoes

qual_pypdf = verifica_qualidade(texto_pypdf, "PyPDF2")
qual_docling = verifica_qualidade(texto_docling, "Docling")

print("📊 COMPARAÇÃO QUALITATIVA: PyPDF2 vs Docling")
print("="*65)
print(f"{'Verificação':<35} {'PyPDF2':>10} {'Docling':>10}")
print("-"*55)

for chave in qual_pypdf:
    if chave == "Ferramenta":
        continue
    v_pypdf = qual_pypdf[chave]
    v_docling = qual_docling[chave]

    # Formata booleanos com emoji
    def fmt(v):
        if isinstance(v, bool):
            return "✅" if v else "❌"
        return str(v)

    print(f"  {chave:<33} {fmt(v_pypdf):>10} {fmt(v_docling):>10}")

print()
print("🏆 Conclusão: Docling preserva estrutura, tabelas e seções.")
print("   PyPDF2 só funciona bem para PDFs de texto simples.")

---
## 🔗 Célula 6 — Integração Docling + LangChain

O objetivo final é ter **LangChain Documents** prontos para indexação no FAISS/OpenSearch.

In [ ]:
# ============================================================
# INTEGRAÇÃO: Docling → LangChain Documents
# ============================================================
from langchain.schema import Document
from langchain.text_splitter import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from pathlib import Path

def pdf_para_langchain_documents(caminho_pdf: str, metadados_extras: dict = None) -> list:
    """
    Pipeline completo: PDF → Docling → Markdown → Chunks LangChain.

    Args:
        caminho_pdf: Caminho para o arquivo PDF
        metadados_extras: Metadados adicionais a incluir em cada chunk

    Returns:
        Lista de LangChain Documents prontos para indexação
    """
    print(f"📄 Processando: {Path(caminho_pdf).name}")

    # PASSO 1: Converter PDF com Docling
    result = converter.convert(caminho_pdf)
    markdown_text = result.document.export_to_markdown()
    print(f"   ✓ Docling: {len(markdown_text)} chars extraídos")

    # PASSO 2: Chunking baseado em headers (Markdown)
    headers_split = [
        ("#", "Seção"),
        ("##", "Subseção"),
        ("###", "Item"),
    ]
    header_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_split,
        strip_headers=False
    )
    chunks_header = header_splitter.split_text(markdown_text)

    # PASSO 3: Para chunks ainda grandes, aplica recursive split
    rec_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )

    documents_finais = []
    for chunk_doc in chunks_header:
        if len(chunk_doc.page_content) > 1200:
            # Chunk muito grande — subdividir
            sub_chunks = rec_splitter.split_text(chunk_doc.page_content)
            for j, sub in enumerate(sub_chunks):
                meta = chunk_doc.metadata.copy()
                meta["sub_chunk"] = j
                if metadados_extras:
                    meta.update(metadados_extras)
                meta["fonte"] = caminho_pdf
                meta["chars"] = len(sub)
                documents_finais.append(Document(page_content=sub, metadata=meta))
        else:
            meta = chunk_doc.metadata.copy()
            if metadados_extras:
                meta.update(metadados_extras)
            meta["fonte"] = caminho_pdf
            meta["chars"] = len(chunk_doc.page_content)
            documents_finais.append(Document(
                page_content=chunk_doc.page_content,
                metadata=meta
            ))

    print(f"   ✓ Chunks gerados: {len(documents_finais)}")
    return documents_finais


# Testando o pipeline completo em ambos os PDFs
docs_acordao = pdf_para_langchain_documents(
    caminho_pdf_simples,
    metadados_extras={"tipo": "acórdão", "tribunal": "TJSP", "ano": 2025}
)

docs_relatorio = pdf_para_langchain_documents(
    caminho_pdf_complexo,
    metadados_extras={"tipo": "relatório_inteligência", "orgao": "PC", "ano": 2025}
)

# Combinar todos os documentos
todos_docs = docs_acordao + docs_relatorio

print()
print(f"📚 Total de Documents prontos para indexação: {len(todos_docs)}")
print()
print("Amostra do primeiro Document:")
print(f"  Metadados: {todos_docs[0].metadata}")
print(f"  Conteúdo: {todos_docs[0].page_content[:200]!r}")

In [ ]:
# ============================================================
# BÔNUS: PDFs Escaneados com OCR
# Demonstração de configuração (não executa OCR real aqui)
# ============================================================
print("🖥️ CONFIGURAÇÃO PARA PDFs ESCANEADOS (OCR)")
print("="*60)
print()
print("Para PDFs digitalizados (imagens), use:")
print()

codigo_ocr = '''
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PipelineOptions
from docling.backend.pypdfium2_backend import PyPdfium2DocumentBackend

# Configuração com OCR habilitado
pipeline_options = PipelineOptions(
    do_ocr=True,                    # ← Habilita OCR
    do_table_structure=True,
    ocr_options={
        "lang": ["pt", "en"],       # ← Idiomas para OCR
        "force_full_page_ocr": False # ← False: OCR só quando necessário
    }
)

converter_ocr = DocumentConverter(
    pipeline_options=pipeline_options
)

# Processar PDF escaneado
result = converter_ocr.convert("laudo_pericial_escaneado.pdf")
texto = result.document.export_to_markdown()
'''

print(codigo_ocr)
print("⚠️  OCR é ~10x mais lento que extração de texto nativo.")
print("   Use apenas quando necessário.")
print()
print("📌 Casos de uso no contexto jurídico:")
print("   - Boletins de ocorrência antigos (pré-2010)")
print("   - Laudos periciais digitalizados")
print("   - Processos físicos escaneados")
print("   - Contratos assinados manualmente e digitalizados")

In [ ]:
# ============================================================
# BÔNUS: Processamento em Lote (múltiplos PDFs)
# Padrão comum em produção jurídica
# ============================================================
from pathlib import Path
from typing import List

def processar_pasta_pdfs(pasta: str, metadados_globais: dict = None) -> List:
    """
    Processa todos os PDFs de uma pasta em lote.

    Em produção, você pode adaptar para:
    - Processar em paralelo (concurrent.futures)
    - Cachear resultados já processados
    - Registrar erros em log
    """
    pasta_path = Path(pasta)
    pdfs = list(pasta_path.glob("*.pdf"))

    if not pdfs:
        print(f"⚠️ Nenhum PDF encontrado em: {pasta}")
        return []

    todos_documentos = []
    erros = []

    print(f"📂 Processando {len(pdfs)} PDFs...")

    for i, pdf_path in enumerate(pdfs, 1):
        print(f"[{i}/{len(pdfs)}] {pdf_path.name}")
        try:
            docs = pdf_para_langchain_documents(
                str(pdf_path),
                metadados_extras={**(metadados_globais or {}), "arquivo": pdf_path.name}
            )
            todos_documentos.extend(docs)
        except Exception as e:
            print(f"   ❌ Erro: {e}")
            erros.append({"arquivo": pdf_path.name, "erro": str(e)})

    print()
    print(f"✅ Processamento concluído:")
    print(f"   PDFs processados: {len(pdfs) - len(erros)}")
    print(f"   Erros: {len(erros)}")
    print(f"   Total de chunks: {len(todos_documentos)}")

    if erros:
        print(f"\n❌ Arquivos com erro:")
        for e in erros:
            print(f"   {e['arquivo']}: {e['erro']}")

    return todos_documentos


# Demonstração com a pasta /tmp
print("🔬 Teste do processamento em lote:")
docs_lote = processar_pasta_pdfs("/tmp")
print(f"\n📊 Total de Documents gerados: {len(docs_lote)}")

---
## ✅ Resumo — Docling vs PyPDF2

| Capacidade | PyPDF2 | Docling |
|---|---|---|
| Texto simples | ✅ Ótimo | ✅ Ótimo |
| Tabelas | ❌ Ilegível | ✅ Estruturado |
| Headers/Seções | ❌ Não detecta | ✅ Markdown |
| PDFs escaneados | ❌ Vazio | ✅ Com OCR |
| DOCX, PPTX, HTML | ❌ Não suporta | ✅ Suportado |
| Velocidade | ⚡⚡⚡ Muito rápido | ⚡⚡ Moderado |
| Instalação | `pip install PyPDF2` | `pip install docling` (+modelos) |

**Quando usar PyPDF2:** PDFs de texto puro, alto volume, sem tabelas, sem estrutura.

**Quando usar Docling:** PDFs com tabelas, estrutura de seções, PDFs escaneados, alta qualidade de extração necessária.

**Próximo notebook:** `LAB1_Naive_RAG_Juridico.ipynb`